# Etapa 2 — Aplicación del modelo sobre `casosNuevos`

**TPI Ciencia de Datos — DS Airlines**

Este notebook aplica el árbol de decisión final (`max_depth=5`, `criterion='gini'`, `random_state=42`) seleccionado en la Etapa 1 sobre el dataset `casosNuevos.xlsx`, con dos objetivos:

1. **Clasificar los 5000 vuelos nuevos** como demorados o no demorados.
2. **Identificar a los clientes que viajan en vuelos clasificados como demorados**, generando el dataset de entrada para las técnicas de clustering y clasificación de la Etapa 2.

Se replica de forma idéntica el pipeline de preparación de datos definido en la Etapa 1 (eliminación de variables, conversión de `hora_salida_programada`, dummy coding con referencias *Lunes / Despejado / Baja*, escalado min-max).

## 1. Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler

# Reproducibilidad
RANDOM_STATE = 42

# Si se ejecuta en Colab, montar Google Drive o subir los archivos manualmente.
# from google.colab import files
# files.upload()

## 2. Carga del dataset de entrenamiento y reentrenamiento del árbol final

Se reentrena el modelo final exactamente como fue definido en la Etapa 1, usando el dataset completo de `Vuelos.xlsx` y los mismos parámetros. Esto asegura que las predicciones sobre `casosNuevos` correspondan al modelo seleccionado.

In [ ]:
# Carga del dataset de entrenamiento
vuelos = pd.read_excel('Vuelos.xlsx')
print(f'Dataset de entrenamiento: {vuelos.shape}')
vuelos.head(3)

Dataset de entrenamiento: (15000, 15)


,id_vuelo,aeropuerto_origen,aeropuerto_destino,hora_salida_programada,dia_semana,distancia_vuelo,condiciones_climaticas,congestion_aerea,tipo_avion,ocupacion_vuelo,temporada_alta,puerta_embarque,visibilidad,tiempo_estimado_vuelo,demora
0,10002,GIG,GRU,09:30,Miércoles,360,Despejado,Baja,Boeing 737,52.8,False,45,12.2,57,1
1,10003,TUC,EZE,11:00,Sábado,1230,Despejado,Alta,Embraer E195,54.4,True,11,10.7,132,1
2,10005,MEX,PTY,08:15,Viernes,2280,Niebla,Media,Boeing 757,48.7,True,7,1.0,199,0


## 3. Pipeline de preparación de datos (Etapa 1)

Se encapsula la transformación en una función para garantizar que entrenamiento y casos nuevos reciban exactamente el mismo tratamiento.

**Pasos del pipeline:**

- Eliminación de variables no utilizadas: `aeropuerto_origen`, `aeropuerto_destino`, `tipo_avion`, `puerta_embarque` y `tiempo_estimado_vuelo` (esta última por r ≈ 0.999 con `distancia_vuelo`).
- Conversión de `hora_salida_programada` de formato `HH:MM` a decimal.
- Conversión de `temporada_alta` a entero 0/1.
- Dummy coding de `dia_semana`, `condiciones_climaticas` y `congestion_aerea`.
- Eliminación de las columnas de referencia: `Lunes`, `Despejado` y `Baja`.

In [ ]:
def hora_a_decimal(h):
    """Convierte una hora en formato 'HH:MM' a decimal."""
    if isinstance(h, str):
        hh, mm = h.split(':')
        return int(hh) + int(mm) / 60
    return h


def preparar_vista_minable(df):
    """
    Aplica el preprocesamiento de la Etapa 1.
    El parámetro `df` puede ser el dataset de entrenamiento (con 'demora')
    o el de casos nuevos (sin 'demora').
    Devuelve la vista minable lista para escalar.
    """
    df = df.copy()

    # Variables excluidas según decisiones de la Fase 3
    cols_excluidas = ['aeropuerto_origen', 'aeropuerto_destino', 'tipo_avion',
                      'puerta_embarque', 'tiempo_estimado_vuelo']
    df = df.drop(columns=cols_excluidas)

    # Transformaciones numéricas
    df['hora_salida_programada'] = df['hora_salida_programada'].apply(hora_a_decimal)
    df['temporada_alta'] = df['temporada_alta'].astype(int)

    # Codificación dummy
    df = pd.get_dummies(df, columns=['dia_semana', 'condiciones_climaticas',
                                     'congestion_aerea'], drop_first=False, dtype=int)

    # Eliminación de categorías de referencia
    referencias = ['dia_semana_Lunes', 'condiciones_climaticas_Despejado',
                   'congestion_aerea_Baja']
    df = df.drop(columns=referencias, errors='ignore')

    return df

In [ ]:
# Aplicar pipeline a entrenamiento
vuelos_prep = preparar_vista_minable(vuelos)

# Separar X e y conservando id_vuelo aparte
id_vuelo_train = vuelos_prep['id_vuelo'].copy()
X_train = vuelos_prep.drop(columns=['id_vuelo', 'demora'])
y_train = vuelos_prep['demora']

print(f'Vista minable de entrenamiento: {X_train.shape}')
print(f'Columnas: {list(X_train.columns)}')

Vista minable de entrenamiento: (15000, 17)
Columnas: ['hora_salida_programada', 'distancia_vuelo', 'ocupacion_vuelo', 'temporada_alta', 'visibilidad', 'dia_semana_Domingo', 'dia_semana_Jueves', 'dia_semana_Martes', 'dia_semana_Miércoles', 'dia_semana_Sábado', 'dia_semana_Viernes', 'condiciones_climaticas_Lluvia', 'condiciones_climaticas_Niebla', 'condiciones_climaticas_Nublado', 'condiciones_climaticas_Tormenta', 'congestion_aerea_Alta', 'congestion_aerea_Media']


## 4. Escalado min-max y entrenamiento del árbol final

Se ajusta el escalador sobre las variables numéricas y se reentrena el árbol final con todos los registros disponibles (no se vuelve a hacer split, porque el modelo ya fue evaluado en la Etapa 1; aquí se utiliza todo el dataset para maximizar la información disponible al predecir sobre `casosNuevos`).

In [ ]:
cols_numericas = ['hora_salida_programada', 'distancia_vuelo', 'ocupacion_vuelo', 'visibilidad']

scaler = MinMaxScaler()
X_train_scaled = X_train.copy()
X_train_scaled[cols_numericas] = scaler.fit_transform(X_train[cols_numericas])

# Reentrenamiento del árbol final
arbol_final = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    random_state=RANDOM_STATE
)
arbol_final.fit(X_train_scaled, y_train)

print('Árbol final reentrenado sobre el dataset completo.')
print(f'  Profundidad: {arbol_final.get_depth()} | Hojas: {arbol_final.get_n_leaves()}')

Árbol final reentrenado sobre el dataset completo.
  Profundidad: 5 | Hojas: 32


## 5. Carga de `casosNuevos.xlsx`

El archivo contiene dos hojas:

- **`VuelosNuevos`**: 5000 vuelos a clasificar (mismas variables que `Vuelos.xlsx`, sin la columna `demora`).
- **`VuelosNuevos_Clientes`**: tabla relacional que vincula cada vuelo con los identificadores de los clientes que viajan en él.

In [ ]:
vuelos_nuevos = pd.read_excel('casosNuevos.xlsx', sheet_name='VuelosNuevos')
rel_vuelo_cliente = pd.read_excel('casosNuevos.xlsx', sheet_name='VuelosNuevos_Clientes')

print(f'Vuelos a clasificar:        {vuelos_nuevos.shape}')
print(f'Relación vuelo-cliente:     {rel_vuelo_cliente.shape}')
vuelos_nuevos.head(3)

Vuelos a clasificar:        (5000, 14)
Relación vuelo-cliente:     (356416, 2)


,id_vuelo,aeropuerto_origen,aeropuerto_destino,hora_salida_programada,dia_semana,distancia_vuelo,condiciones_climaticas,congestion_aerea,tipo_avion,ocupacion_vuelo,temporada_alta,puerta_embarque,visibilidad,tiempo_estimado_vuelo
0,20651,BOG,MEX,15:45,Lunes,2700,Despejado,Media,Boeing 757,62.4,True,42,21.1,231
1,12042,MDE,JFK,06:45,Jueves,3560,Despejado,Alta,Airbus A350,79.7,False,45,17.8,310
2,18669,BRC,EZE,17:00,Sábado,1650,Nublado,Media,Boeing 737 MAX,55.2,False,24,8.3,164


## 6. Aplicación del pipeline sobre los casos nuevos

Se aplica la misma función `preparar_vista_minable` y luego se **alinean las columnas** con las usadas en entrenamiento mediante `reindex`. Esto garantiza que, ante cualquier diferencia en categorías presentes, las columnas dummy queden en el mismo orden y con los mismos nombres que el modelo espera.

El escalado se realiza con el `MinMaxScaler` **ya ajustado** en entrenamiento (`transform`, no `fit_transform`), conforme a la buena práctica de evitar fugas de información.

In [ ]:
# Aplicar pipeline
nuevos_prep = preparar_vista_minable(vuelos_nuevos)

# Conservar id_vuelo aparte
id_vuelo_nuevos = nuevos_prep['id_vuelo'].copy()
X_nuevos = nuevos_prep.drop(columns=['id_vuelo'])

# Alineación de columnas con las del modelo entrenado
X_nuevos = X_nuevos.reindex(columns=X_train.columns, fill_value=0)

# Escalado con el scaler ya ajustado
X_nuevos_scaled = X_nuevos.copy()
X_nuevos_scaled[cols_numericas] = scaler.transform(X_nuevos[cols_numericas])

print(f'Vista minable de casos nuevos: {X_nuevos_scaled.shape}')
print(f'Columnas coinciden con entrenamiento: {list(X_nuevos_scaled.columns) == list(X_train.columns)}')

Vista minable de casos nuevos: (5000, 17)
Columnas coinciden con entrenamiento: True


## 7. Predicción de demora sobre los vuelos nuevos

In [ ]:
predicciones = arbol_final.predict(X_nuevos_scaled)

# Tabla final de vuelos clasificados (con todos los atributos originales + predicción)
vuelos_clasificados = vuelos_nuevos.copy()
vuelos_clasificados['predicho_demora'] = predicciones

# Distribución de la predicción
print('Distribución de la predicción sobre casos nuevos:')
dist = pd.Series(predicciones).value_counts(normalize=True).round(4) * 100
print(dist.to_string())
print(f'\nTotal vuelos clasificados como demorados: {(predicciones == 1).sum()}')
print(f'Total vuelos clasificados como no demorados: {(predicciones == 0).sum()}')

Distribución de la predicción sobre casos nuevos:
0    86.82
1    13.18

Total vuelos clasificados como demorados: 659
Total vuelos clasificados como no demorados: 4341


In [ ]:
vuelos_clasificados.head()

,id_vuelo,aeropuerto_origen,aeropuerto_destino,hora_salida_programada,dia_semana,distancia_vuelo,condiciones_climaticas,congestion_aerea,tipo_avion,ocupacion_vuelo,temporada_alta,puerta_embarque,visibilidad,tiempo_estimado_vuelo,predicho_demora
0,20651,BOG,MEX,15:45,Lunes,2700,Despejado,Media,Boeing 757,62.4,True,42,21.1,231,0
1,12042,MDE,JFK,06:45,Jueves,3560,Despejado,Alta,Airbus A350,79.7,False,45,17.8,310,0
2,18669,BRC,EZE,17:00,Sábado,1650,Nublado,Media,Boeing 737 MAX,55.2,False,24,8.3,164,0
3,11115,MIA,BOG,07:30,Sábado,3160,Despejado,Media,Boeing 787,66.6,False,19,7.1,266,0
4,23903,MDE,UIO,14:30,Miércoles,960,Nublado,Baja,Embraer E195,73.3,False,30,18.8,104,0


## 8. Identificación de clientes que viajan en vuelos demorados

Se filtran los vuelos predichos como demorados y se cruzan con la tabla relacional para obtener los `id_cliente` afectados. Luego se cruza con `Clientes.xlsx` para recuperar todos los atributos de esos clientes, generando el dataset de entrada para los modelos de la Etapa 2.

In [ ]:
# Vuelos clasificados como demorados
vuelos_demorados = vuelos_clasificados[vuelos_clasificados['predicho_demora'] == 1].copy()
print(f'Vuelos demorados: {len(vuelos_demorados)}')

# Cruce con la tabla relacional para obtener los id_cliente afectados
rel_demorados = rel_vuelo_cliente.merge(
    vuelos_demorados[['id_vuelo']],
    on='id_vuelo',
    how='inner'
)
print(f'Registros vuelo-cliente sobre vuelos demorados: {len(rel_demorados)}')
print(f'Clientes únicos afectados: {rel_demorados["id_cliente"].nunique()}')

Vuelos demorados: 659
Registros vuelo-cliente sobre vuelos demorados: 47247
Clientes únicos afectados: 30518


In [ ]:
# Carga de Clientes.xlsx
clientes = pd.read_excel('Clientes.xlsx')
print(f'Dataset de clientes: {clientes.shape}')
clientes.head(3)

Dataset de clientes: (50000, 14)


,id_cliente,sexo,provincia,edad,ocupacion,cant_vuelos,clase_preferida,gasto_acumulado,gasto_acumulado_extra,programaMillas,cantidad_millas,ingreso_mensual,anticipacion_compra_promedio,canal_compra
0,1,M,Salta,24,Comerciante,21,Económica,5731.62,1222.62,Sí,19113,1367,3,Agencia
1,2,F,Córdoba,49,Médico,16,Económica,2593.40,510.70,No,0,6252,30,App
2,3,M,Buenos Aires,41,Docente,10,Económica,2850.62,627.01,No,0,1593,16,App


In [ ]:
# Lista deduplicada de clientes en vuelos demorados (input para Etapa 2)
ids_clientes_demorados = rel_demorados['id_cliente'].drop_duplicates()

clientes_a_analizar = clientes.merge(
    ids_clientes_demorados.to_frame(),
    on='id_cliente',
    how='inner'
)

print(f'Clientes a analizar en Etapa 2: {clientes_a_analizar.shape}')
clientes_a_analizar.head()

Clientes a analizar en Etapa 2: (30518, 14)


,id_cliente,sexo,provincia,edad,ocupacion,cant_vuelos,clase_preferida,gasto_acumulado,gasto_acumulado_extra,programaMillas,cantidad_millas,ingreso_mensual,anticipacion_compra_promedio,canal_compra
0,1,M,Salta,24,Comerciante,21,Económica,5731.62,1222.62,Sí,19113,1367,3,Agencia
1,2,F,Córdoba,49,Médico,16,Económica,2593.40,510.70,No,0,6252,30,App
2,3,M,Buenos Aires,41,Docente,10,Económica,2850.62,627.01,No,0,1593,16,App
3,4,F,São Paulo,19,Empleado,9,Económica,2749.41,540.22,No,0,862,47,Agencia
4,5,M,Rio Grande do Sul,31,Profesional independiente,16,Económica,5751.04,817.55,No,0,2927,26,Web


## 9. Exportación de resultados

Se genera un único archivo Excel con cuatro hojas:

- **`vuelos_clasificados`**: los 5000 vuelos con su predicción.
- **`vuelos_demorados`**: subset con los vuelos clasificados como demorados.
- **`clientes_a_analizar`**: clientes (sin duplicados) que viajan en al menos un vuelo demorado, con todos sus atributos. **Este es el dataset de entrada para los modelos de la Etapa 2.**
- **`relacion_demorados`**: tabla relacional filtrada, necesaria para reconstruir luego la asignación de beneficios por pasajero.

In [ ]:
archivo_salida = 'resultados_etapa2_arbol.xlsx'

with pd.ExcelWriter(archivo_salida, engine='openpyxl') as writer:
    vuelos_clasificados.to_excel(writer, sheet_name='vuelos_clasificados', index=False)
    vuelos_demorados.to_excel(writer, sheet_name='vuelos_demorados', index=False)
    clientes_a_analizar.to_excel(writer, sheet_name='clientes_a_analizar', index=False)
    rel_demorados.to_excel(writer, sheet_name='relacion_demorados', index=False)

print(f'Archivo generado: {archivo_salida}')
print('\nResumen:')
print(f'  - vuelos_clasificados:                {len(vuelos_clasificados)} filas')
print(f'  - vuelos_demorados:                   {len(vuelos_demorados)} filas')
print(f'  - clientes_a_analizar:                {len(clientes_a_analizar)} filas')
print(f'  - relacion_demorados:                 {len(rel_demorados)} filas')

Archivo generado: resultados_etapa2_arbol.xlsx

Resumen:
  - vuelos_clasificados:                5000 filas
  - vuelos_demorados:                   659 filas
  - clientes_a_analizar:                30518 filas
  - relacion_demorados:                 47247 filas


## 10. Conclusión

El árbol de decisión final clasificó cada uno de los 5000 vuelos de `casosNuevos`, y a partir de los vuelos predichos como demorados se construyó la lista de clientes a los que deberá ofrecerse algún beneficio (voucher o sala VIP).

El dataset `clientes_a_analizar` será el insumo de la próxima fase del trabajo, donde se aplicarán técnicas de clustering y clasificación para caracterizar perfiles de pasajeros y decidir qué tipo de beneficio asignar a cada uno. La tabla `relacion_vuelo_cliente_demorados` se conservará para, una vez obtenidos los perfiles, reconstruir la tabla final de asignación pasajero–vuelo–beneficio solicitada en el enunciado.